首先单元测试生成 infer，debug 完这一块的pipeline

加入track

完成整个加载-生成-评估-保存

In [6]:
# 启用自动重载扩展，使得在修改Python模块后不需要重启内核就能使用最新的代码
%load_ext autoreload

# 设置自动重载模式为2，表示所有模块都会被自动重载
# 这在开发过程中非常有用，因为不需要手动重启kernel来应用代码更改
%autoreload 2

In [7]:
# 配置

import torch
from shepherd.lightning_module import LightningModule


chkpt = '/home1/zhh/workspace/SPD/training/jobs/x1x3x4_diffusion_mosesaq_20240824/last.ckpt'
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model_pl = LightningModule.load_from_checkpoint(chkpt) #
params = model_pl.params
model_pl.to(device)
model_pl.model.device = device

In [8]:
# ===== 分子数据加载模块 =====
# 算法核心思想：从预处理的分子数据库中加载标准化的分子结构和电荷信息
# 实现原理：
# 1. 数据源管理：使用pickle格式存储的分子数据库，包含MolBlock格式的3D结构信息
# 2. 结构标准化：每个分子包含原子坐标、键连接信息和部分电荷数据
# 3. 批量处理：支持同时加载多个分子用于后续的条件生成和评估
import pickle

# 分子数据加载算法
# 功能：从标准化的分子数据库中提取结构和电荷信息
# 输入：预处理的pkl文件，包含MolBlock格式的分子结构数据
# 输出：包含3个分子的结构和电荷信息的数据集
# 数据格式：每个分子包含MolBlock字符串和对应的原子部分电荷数组
with open('/home1/zhh/workspace/SPD/data/conformers/np/molblock_charges_NPs.pkl', 'rb') as f:
    # 从pkl文件中读取molblock和charges数据
    molblocks_and_charges = pickle.load(f)
    # 打印数据长度以确认实际包含的分子数量
    print(f"加载的数据包含 {len(molblocks_and_charges)} 个分子")

# 选择要处理的天然产物分子的索引
index = 0 # 0, 1, 2

加载的数据包含 3 个分子


In [9]:
import rdkit
import numpy as np

# 从molblock创建RDKit分子对象,保留氢原子
mol = rdkit.Chem.MolFromMolBlock(molblocks_and_charges[index][0], removeHs = False) # target natural product
# 获取分子的隐式水环境下的xTB部分电荷
charges = np.array(molblocks_and_charges[index][1]) # xTB partial charges in implicit water
# 使用display函数显示RDKit分子对象mol的2D结构图
display(mol)
# 返回是 <rdkit.Chem.rdchem.Mol at 0x7f3412458cf0> 表示一个RDKit分子对象的内存地址信息

In [10]:
mol_block_string = rdkit.Chem.MolToMolBlock(mol)
print("分子的MolBlock信息如下: ")
print(mol_block_string)

# 遗憾不可以生成分子图片
# from rdkit.Chem import Draw
# # 将mol对象直接转换为一张图片
# mol_image = Draw.MolToImage(mol)
# # 在Jupyter Notebook中，只需在代码块的最后一行写上变量名，它就会自动显示图片
# mol_image

# ImportError: libXrender.so.1: cannot open shared object file: No such file or directory 错误
# 不是Python代码本身的问题，而是你的操作系统里缺少了一个重要的“零件”。
# 但是我没有sudo权限

分子的MolBlock信息如下: 

     RDKit          3D

 78 81  0  0  0  0  0  0  0  0999 V2000
    0.8507    4.5925    1.2131 C   0  0  0  0  0  0  0  0  0  0  0  0
    0.8202    4.3798   -0.2956 C   0  0  0  0  0  0  0  0  0  0  0  0
    0.6403    2.8992   -0.6517 C   0  0  1  0  0  0  0  0  0  0  0  0
   -0.7191    2.4056   -0.2631 C   0  0  0  0  0  0  0  0  0  0  0  0
   -0.9445    1.4116    0.5779 C   0  0  0  0  0  0  0  0  0  0  0  0
   -2.3067    0.8421    0.8502 C   0  0  2  0  0  0  0  0  0  0  0  0
   -2.3898    0.4432    2.1954 O   0  0  0  0  0  0  0  0  0  0  0  0
   -2.5578   -0.3344   -0.1299 C   0  0  2  0  0  0  0  0  0  0  0  0
   -3.7970   -1.1794    0.1840 C   0  0  0  0  0  0  0  0  0  0  0  0
   -5.1436   -0.4374    0.2155 C   0  0  0  0  0  0  0  0  0  0  0  0
   -5.3659    0.3871   -1.0570 C   0  0  0  0  0  0  0  0  0  0  0  0
   -6.2632   -1.4766    0.3586 C   0  0  0  0  0  0  0  0  0  0  0  0
   -5.1228    0.4035    1.3648 O   0  0  0  0  0  0  0  0  0  0  0  0
   -1.4

In [11]:
# ===== 分子坐标预处理算法 =====
# 算法核心思想：对分子3D坐标进行标准化处理，为后续特征提取做准备
# 实现原理：
# 1. 坐标提取：从RDKit分子对象中获取原子的三维空间坐标
# 2. 中心化处理：将分子几何中心移至原点，消除平移变换的影响
# 3. 坐标更新：将处理后的坐标重新赋值给分子对象
from shepherd.shepherd_score_utils.conformer_generation import update_mol_coordinates

# 分子坐标标准化算法
# 步骤1：提取原始3D坐标矩阵 (N×3，N为原子数)
mol_coordinates = np.array(mol.GetConformer().GetPositions())

# 步骤2：计算几何中心并进行中心化处理
# 目的：消除分子在空间中的绝对位置影响，只保留相对几何结构
mol_coordinates = mol_coordinates - np.mean(mol_coordinates, axis = 0)

# 步骤3：更新分子对象的坐标信息
# 功能：将标准化后的坐标重新写入RDKit分子对象
mol = update_mol_coordinates(mol, mol_coordinates)


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [12]:
# ===== 多模态条件特征提取算法 =====
# 算法核心思想：从分子3D结构中提取用于条件生成的多模态特征
# 实现原理：
# 1. 几何特征提取：计算分子表面点云和范德华半径
# 2. 电子特征计算：基于原子部分电荷计算表面静电势分布
# 3. 药效团识别：提取分子的药效团类型、位置和方向信息
# 4. 多模态融合：将几何、电子和药效团特征整合为条件输入
from shepherd.shepherd_score_utils.generate_point_cloud import (
    get_atomic_vdw_radii, 
    get_molecular_surface,
    get_electrostatics_given_point_charges,
)
from shepherd.shepherd_score_utils.pharm_utils.pharmacophore import get_pharmacophores

# 条件特征提取算法工作流程
# 步骤1：提取原子几何信息
# 功能：获取所有原子的三维坐标作为几何基础
centers = mol.GetConformer().GetPositions()
print("原子中心坐标 centers shape:", centers.shape)

# 步骤2：计算原子范德华半径
# 功能：获取每个原子的范德华半径，用于分子表面计算
radii = get_atomic_vdw_radii(mol)
print("范德华半径 radii shape:", radii.shape)

# 步骤3：生成分子表面点云 (x3模态)
# 算法：基于原子坐标和范德华半径，使用探针球滚动算法生成分子可及表面
# 参数：num_points控制表面点密度，probe_radius为探针半径，num_samples_per_atom为每原子采样数
surface = get_molecular_surface(
    centers, 
    radii, 
    params['dataset']['x3']['num_points'], # 表面点数量
    probe_radius = params['dataset']['probe_radius'], # 探针半径
    num_samples_per_atom = 20, # 每个原子的采样点数
)
print("分子表面点 surface shape:", surface.shape)

# 步骤4：提取药效团特征 (x4模态)
# 算法：基于RDKit的药效团识别算法，提取氢键供体/受体、疏水区域等特征
# 输出：药效团类型、三维位置坐标和方向向量
pharm_types, pharm_pos, pharm_direction = get_pharmacophores(
    mol,
    multi_vector = params['dataset']['x4']['multivectors'], # 是否使用多向量表示
    check_access = params['dataset']['x4']['check_accessibility'], # 是否检查可及性
)
print("药效团类型 pharm_types:", pharm_types.shape)
print("药效团位置 pharm_pos shape:", pharm_pos.shape)
print("药效团方向 pharm_direction shape:", pharm_direction.shape)

# 步骤5：计算表面静电势分布
# 算法：基于库仑定律，使用原子部分电荷计算表面各点的静电势值
# 公式：V(r) = Σ(qi / |r - ri|)，其中qi为原子电荷，ri为原子位置
electrostatics = get_electrostatics_given_point_charges(
    charges, centers, surface,
)
print("静电势 electrostatics shape:", electrostatics.shape)


原子中心坐标 centers shape: (78, 3)
范德华半径 radii shape: (78,)
分子表面点 surface shape: (75, 3)
药效团类型 pharm_types: (19,)
药效团位置 pharm_pos shape: (19, 3)
药效团方向 pharm_direction shape: (19, 3)
静电势 electrostatics shape: (75,)


In [13]:
# ===== 条件生成模型配置参数 =====
# 算法核心：控制条件分子生成过程的关键超参数设置
# 实现原理：
# 1. 分子规模控制：通过n_atoms限制生成分子的复杂度和计算开销
# 2. 并行处理优化：通过batch_size平衡内存使用和生成效率
# 3. 条件约束匹配：确保药效团数量与条件输入的维度一致性

# 参数1：目标分子原子数量
# 功能：定义生成分子的最大原子数，影响分子复杂度和生成质量
# 取值范围：通常10-100，根据目标分子类型调整
n_atoms = 70

# 参数2：批处理大小
# 功能：控制同时生成的分子数量，影响内存占用和计算效率
# 优化策略：根据GPU内存容量和生成速度需求平衡设置
batch_size = 5

# 参数3：药效团特征维度
# 算法约束：在条件生成(inpainting)模式下，药效团数量必须与输入条件的药效团位置数量严格匹配
# 数据一致性：确保pharm_types长度等于pharm_pos.shape[0]，维持条件-生成的对应关系
num_pharmacophores = len(pharm_types) # must equal pharm_pos.shape[0] if inpainting

# 计算数据集的边际分布

In [14]:
# ===== 数据集边际分布统计算法 =====
# 算法核心思想：统计训练数据中各类特征的出现频率，为生成模型提供先验分布
# 实现原理：
# 1. 特征类型定义：基于化学知识定义原子类型、键类型和药效团类型
# 2. 频率统计：遍历数据集统计每种特征类型的出现次数
# 3. 概率分布：将计数转换为概率分布，指导生成过程的采样
# 4. 先验知识：为扩散模型提供化学合理的特征分布先验
from tqdm import tqdm
from shepherd.shepherd_score_utils.pharm_utils.pharmacophore import get_pharmacophores

print("初始化用于统计各类特征出现次数的计数器")

# 边际分布计算的特征类型定义
# 数据来源：training/parameters/params_x1x3x4_diffusion_mosesaq_20240824.py

# 原子类型定义 (x1模态)
# 包含常见的有机分子原子类型，None表示空位或填充
atom_types_x1 = [None, 'H', 'C', 'N', 'O', 'F', 'Cl', 'Br', 'I', 'S', 'P', 'Si']

# 化学键类型定义 (x1模态)
# 涵盖所有主要的化学键类型，用于分子图表示
bond_types_x1 = [None, 'SINGLE', 'DOUBLE', 'TRIPLE', 'AROMATIC']

# 药效团类型最大数量 (x4模态)
# 定义药效团特征的维度上限
max_node_types_x4 = 10

# 初始化特征计数器
# 算法：使用torch.zeros创建计数张量，便于后续的GPU加速计算
# 数据类型：float类型支持概率计算和梯度传播
atom_counts = torch.zeros(len(atom_types_x1), dtype=torch.float)
bond_counts = torch.zeros(len(bond_types_x1), dtype=torch.float)
pharm_counts = torch.zeros(max_node_types_x4, dtype=torch.float)

# 化学键类型映射函数
# 功能：将RDKit的键类型对象转换为参数文件中使用的字符串表示
def get_bond_type_str(bond):
    return str(bond.GetBondType())

# 边际分布统计主循环
# 算法流程：遍历整个数据集，统计各类特征的出现次数
for mol_block, _ in tqdm(molblocks_and_charges, desc="Counting feature occurrences"):
    # 步骤1：从MolBlock字符串创建RDKit分子对象
    # 保留氢原子：removeHs=False确保统计包含所有原子类型
    mol = rdkit.Chem.MolFromMolBlock(mol_block, removeHs=False)
    if not mol:
        print("Warning: Failed to create molecule from MolBlock. 跳过了奥.")
        continue
    
    # 步骤2：统计原子类型出现次数
    # 遍历分子中的每个原子，累加对应类型的计数
    for atom in mol.GetAtoms():
        symbol = atom.GetSymbol()
        if symbol in atom_types_x1:
            atom_counts[atom_types_x1.index(symbol)] += 1

    # 步骤3：统计化学键类型出现次数
    # 遍历分子中的每个化学键，累加对应类型的计数
    for bond in mol.GetBonds():
        bond_str = get_bond_type_str(bond)
        if bond_str in bond_types_x1:
            bond_counts[bond_types_x1.index(bond_str)] += 1
     
    try:
        # 步骤4：统计药效团类型出现次数
        # 算法：使用RDKit的药效团识别功能提取分子的药效团特征
        pharm_types_temp, _, _ = get_pharmacophores(
            mol, 
            multi_vector=False,  # 不使用多向量表示
            check_access=False   # 不检查可及性
        ) 
        # 索引偏移处理：数据集代码为每个药效团类型加1，为虚拟节点预留索引0
        # 必须复制此逻辑以确保计数的准确性
        for p_type in (pharm_types_temp + 1):
            if p_type < max_node_types_x4:
                pharm_counts[p_type] += 1
    except Exception as e:
        # This can happen for molecules that don't have RDKit's feature definitions
        print(f"Warning: Could not get pharmacophores for a molecule. Skipping. Error: {e}")

# --- 3. Normalize counts to get probabilities (marginals) ---

# Handle the case where a feature type might have zero counts to avoid division by zero


atom_marginals_x1 = (atom_counts / atom_counts.sum()) if atom_counts.sum() > 0 else torch.ones_like(atom_counts) / len(atom_counts)
bond_marginals_x1 = (bond_counts / bond_counts.sum()) if bond_counts.sum() > 0 else torch.ones_like(bond_counts) / len(bond_counts)
pharm_marginals_x4 = (pharm_counts / pharm_counts.sum()) if pharm_counts.sum() > 0 else torch.ones_like(pharm_counts) / len(pharm_counts)

print("\n--- 边际分布 计算完毕 ---")
print(f"Atom Marginals (x1): {atom_marginals_x1.shape}")
print(f"Bond Marginals (x1): {bond_marginals_x1.shape}")
print(f"Pharmacophore Marginals (x4): {pharm_marginals_x4.shape}")
print("---------------------------------------\n")


初始化用于统计各类特征出现次数的计数器


Counting feature occurrences: 100%|██████████| 3/3 [00:00<00:00, 37.34it/s]


--- 边际分布 计算完毕 ---
Atom Marginals (x1): torch.Size([12])
Bond Marginals (x1): torch.Size([5])
Pharmacophore Marginals (x4): torch.Size([10])
---------------------------------------



In [15]:
import json
import numpy as np

# 在保存时转换numpy数组为列表
def convert_numpy_to_list(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_numpy_to_list(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_to_list(item) for item in obj]
    return obj


In [16]:
# ===== 条件分子生成推理算法 =====
# 算法核心思想：基于扩散模型的条件分子生成，固定x2(表面)和x3(静电势)条件，生成x1(原子图)和x4(药效团)
# 实现原理：
# 1. 扩散过程：从噪声开始，通过逐步去噪生成目标分子结构
# 2. 条件约束：在生成过程中保持指定条件不变，确保生成结果符合约束
# 3. 多模态融合：同时处理原子图、表面、静电势和药效团四种模态
# 4. 修复机制：通过inpainting技术精确控制各模态的生成行为
from shepherd.inference import *
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

# 条件分子生成主函数调用
# 算法：使用训练好的扩散模型进行条件生成推理
# 调用推理采样函数并将结果保存到文件
generated_samples = inference_sample(
    model_pl,  # 预训练的扩散模型对象
    batch_size = batch_size,  # 批处理大小，控制并行生成数量
    
    # 生成目标的维度控制
    N_x1 = n_atoms,  # x1模态：目标分子的原子数量
    N_x4 = num_pharmacophores,  # x4模态：目标分子的药效团数量
    
    # 生成模式配置
    unconditional = False,  # 条件生成模式：False表示基于给定条件生成
    
    # 噪声控制参数
    # 算法：控制扩散过程中的噪声强度，影响生成质量和多样性
    prior_noise_scale = 1.0,  # 先验噪声缩放：控制初始噪声强度
    denoising_noise_scale = 1.0,  # 去噪噪声缩放：控制去噪过程的噪声水平
    
    # 动态噪声注入机制
    # 功能：在特定时间步注入额外噪声，增加生成多样性
    inject_noise_at_ts = [],  # 噪声注入时间步列表（空表示不注入）
    inject_noise_scales = [],  # 对应的噪声强度列表
    
    # 谐波化处理参数
    # 算法：通过谐波化技术优化生成轨迹，提高生成质量
    harmonize = False,  # 谐波化开关：False表示不启用
    harmonize_ts = [],  # 谐波化时间步列表
    harmonize_jumps = [],  # 谐波化跳跃步长列表
    
    # ===== 条件修复(Inpainting)参数配置 =====
    # 核心机制：精确控制各模态在生成过程中的行为
    
    # x2模态修复配置（原子位置，通过x3隐式建模）
    inpaint_x2_pos = False,  # 不修复x2位置：作为固定条件
    
    # x3模态修复配置（分子表面和静电势）
    inpaint_x3_pos = False,  # 不修复x3位置：作为固定条件输入
    inpaint_x3_x = False,  # 不修复x3特征：保持表面和静电势不变
    
    # x4模态修复配置（药效团特征）
    # 策略：完全修复x4，使其与生成的x1原子图保持一致
    inpaint_x4_pos = True,  # 修复药效团位置：确保空间一致性
    inpaint_x4_direction = True,  # 修复药效团方向：确保方向一致性
    inpaint_x4_type = True,  # 修复药效团类型：确保类型一致性
    
    # ===== 修复过程的时间和噪声控制 =====
    # 算法：通过时间控制和噪声调节优化修复效果
    
    # x2修复时间控制
    stop_inpainting_at_time_x2 = 0.0,  # x2停止修复时间：0.0表示全程修复
    add_noise_to_inpainted_x2_pos = 0.0,  # x2修复噪声：0.0表示无额外噪声
    
    # x3修复时间控制
    stop_inpainting_at_time_x3 = 0.0,  # x3停止修复时间：0.0表示全程修复
    add_noise_to_inpainted_x3_pos = 0.0,  # x3位置修复噪声
    add_noise_to_inpainted_x3_x = 0.0,  # x3特征修复噪声
    
    # x4修复时间控制
    # 策略：全程修复x4各属性，确保与生成的x1完全匹配
    stop_inpainting_at_time_x4 = 0.0,  # x4停止修复时间：0.0表示全程修复
    add_noise_to_inpainted_x4_pos = 0.0,  # x4位置修复噪声：0.0确保精确匹配
    add_noise_to_inpainted_x4_direction = 0.0,  # x4方向修复噪声：0.0确保精确匹配
    add_noise_to_inpainted_x4_type = 0.0,  # x4类型修复噪声：0.0确保精确匹配
    
    # 条件输入数据
    center_of_mass = np.zeros(3),  # x1的质心(已经中心化为零)
    surface = surface,  # 表面特征 x2
    electrostatics = electrostatics,  # 静电特征 x3
    pharm_types = pharm_types,  # 药效团类型 x3
    pharm_pos = pharm_pos,  # 药效团位置
    pharm_direction = pharm_direction,  # 药效团方向

    atom_marginals=atom_marginals_x1,
    bond_marginals=bond_marginals_x1,
)

# generated_structures = []  # 存储生成的结构列表
# for b in range(batch_size):  # 遍历批次中的每个样本
#     generated_dict = {
#         'x1': {  # X1模态：分子结构
#             'atoms': np.array_split(x1_x_final.cpu().numpy(), batch_size)[b],  # 原子类型（原子序数）
#             #'formal_charges': None, # 形式电荷（待提取）：从x1_x_t[~virtual_node_mask_x1, -len(params['dataset']['x1']['charge_types']):]中提取
#             'bonds': np.array_split(x1_bond_edge_x_final, batch_size)[b],  # 键类型
#             'positions': np.array_split(x1_pos_final, batch_size)[b],  # 原子位置坐标
#         },
#         'x2': {  # X2模态：蛋白质口袋结构
#             'positions': np.array_split(surface, batch_size)[b],  # 口袋表面点位置
#         },
#         'x3': {  # X3模态：静电势场
#             'charges': np.array_split(electrostatics, batch_size)[b],  # 静电势值
#             'positions': np.array_split(pharm_pos, batch_size)[b],  # 静电势点位置
#         },
#         'x4': {  # X4模态：药效团特征
#             'types': np.array_split(x4_x_final, batch_size)[b],  # 药效团类型
#             'positions': np.array_split(x4_pos_final, batch_size)[b],  # 药效团位置
#             'directions': np.array_split(x4_direction_final, batch_size)[b],  # 药效团方向向量
#         },
#     }
#     generated_structures.append(generated_dict)  # 添加到结果列表

# return generated_structures  # 返回生成的多模态结构列表

Initialized DiscreteFeatureDiffusion with marginals: tensor([0.0000, 0.4742, 0.3944, 0.0047, 0.1268, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000])
Initialized DiscreteFeatureDiffusion with marginals: tensor([0.0000, 0.8705, 0.0804, 0.0000, 0.0491])


100%|██████████| 400/400 [07:25<00:00,  1.11s/it]


In [18]:
import json
import numpy as np
import torch  # 导入 torch 库，因为我们需要处理 Tensor 类型

# 这是一个升级版的辅助函数，可以同时处理 numpy 数组和 torch Tensor
def convert_for_json(obj):
    """
    功能说明：递归地将一个复杂的数据结构（可能包含字典、列表、numpy数组和torch张量）
              转换为能够被JSON库序列化的纯Python数据结构（只包含字典、列表、字符串、数字等）。
    参数解释：
        obj: 可能是字典、列表、numpy数组、torch张量等任意类型的对象。
    实现逻辑：
        - 如果对象是字典，就递归地转换它的每一个值。
        - 如果对象是列表，就递归地转换它的每一个元素。
        - 如果对象是numpy数组，就使用 .tolist() 方法将其转换为Python列表。
        - 如果对象是torch张量，先用 .cpu() 把它从GPU（如果有的话）移到CPU，
          再用 .numpy() 转成numpy数组，最后用 .tolist() 转成Python列表。
        - 其他类型的对象直接返回。
    注意事项：这个函数假设 torch 库已经安装并可以导入。
    """
    if isinstance(obj, dict):
        return {k: convert_for_json(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [convert_for_json(elem) for elem in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, torch.Tensor):
        return obj.cpu().numpy().tolist()
    return obj

# 假设 generated_samples 是你已经生成好的数据
# 1. 先使用我们新的辅助函数来转换数据
generated_samples_for_json = convert_for_json(generated_samples)

# 2. 保存到 JSON 文件
# 我们把文件名改成 output.json
with open('output.json', 'w', encoding='utf-8') as f:
    # 使用 json.dump 来写入文件
    # indent=4 会让文件内容格式化，看起来更整齐，方便阅读
    json.dump(generated_samples_for_json, f, ensure_ascii=False, indent=4)

print("数据已成功保存到 output.json 文件！")

数据已成功保存到 output.json 文件！


In [20]:
import json
import numpy as np

# 从 JSON 文件读取数据
with open('output.json', 'r', encoding='utf-8') as f:
    loaded_data = json.load(f)

# 现在 loaded_data 的结构和你的 generated_samples 一样，
# 只是里面都是 Python 列表而不是 numpy 数组。
# 我们可以写一个循环把它转换回来。

reloaded_samples = []
for sample in loaded_data:
    # 遍历 'x1', 'x2', 'x3', 'x4' 等模态
    for modal_key in sample:
        # 遍历 'atoms', 'bonds', 'positions' 等数据
        for data_key in sample[modal_key]:
            # 把列表转换回 numpy 数组
            if isinstance(sample[modal_key][data_key], list):
                sample[modal_key][data_key] = np.array(sample[modal_key][data_key])
    reloaded_samples.append(sample)


print("从 output.json 文件中成功读取并恢复了数据！")



从 output.json 文件中成功读取并恢复了数据！


In [21]:
# reloaded_samples 现在就和最开始的 generated_samples 结构和类型都一样了
# 你可以检查一下来确认
print(type(reloaded_samples[0]['x1']['atoms']))

<class 'numpy.ndarray'>


In [22]:
reloaded_samples

[{'x1': {'atoms': array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
          [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
          [0, 1, 0, 0, 0,

# 评估模块 Evaluation

化学合理性

分子生成构象的正确性

与表面输入条件的一致性

与ground-truth分子的分子指纹相似性

多样性



In [23]:
# ===== 评估模块初始化 =====
# 算法核心思想：构建多维度分子生成质量评估体系，包括化学合理性、构象正确性、条件一致性等
# 实现原理：
# 1. 化学合理性评估：基于RDKit验证分子结构的化学有效性
# 2. 构象正确性评估：使用xTB方法计算RMSD评估三维构象质量
# 3. 条件一致性评估：验证生成结果与输入条件的匹配度
# 4. 多样性评估：分析生成分子的结构多样性和覆盖度

# ===== 核心依赖库导入 =====
import open3d  # 3D可视化库，优先导入避免潜在的导入冲突
import numpy as np  # 数值计算库，用于矩阵运算和数据处理
from rdkit import Chem  # 化学信息学库，用于分子结构处理和验证

# ===== 分子构象生成模块 =====
# 功能：从SMILES字符串生成三维分子构象
# 算法：基于MMFF力场优化的构象嵌入算法
from shepherd_score.conformer_generation import embed_conformer_from_smiles

# ===== 评估管道模块 =====
# ConfEval：分子构象评估核心类，提供多种评估指标
# UnconditionalEvalPipeline：无条件生成评估管道
# ConsistencyEvalPipeline：一致性评估管道，验证生成结果的内在一致性
# ConditionalEvalPipeline：条件生成评估管道，验证条件约束的满足程度
from shepherd_score.evaluations.evaluate import ConfEval, UnconditionalEvalPipeline
from shepherd_score.evaluations.evaluate import ConsistencyEvalPipeline, ConditionalEvalPipeline

# 检测二维属性

In [61]:
# ===== 测试分子构象生成算法 =====
# 算法核心思想：从SMILES字符串生成优化的三维分子构象，用于验证评估系统功能
# 实现原理：
# 1. SMILES解析：将一维化学结构字符串转换为分子图表示
# 2. 构象嵌入：使用距离几何算法生成初始三维坐标
# 3. MMFF优化：通过分子力场优化构象能量，获得稳定结构

# 生成测试分子的三维构象
# 输入：复杂有机分子的SMILES字符串（含氯代芳环和杂环结构）
# 算法：MMFF力场优化的构象嵌入
# 输出：包含优化三维坐标的RDKit分子对象
# rdkit_mol = embed_conformer_from_smiles(
#     'c1Cc2ccc(Cl)cc2C(=O)c1c3cc(N1nnc2cc(C)c(Cl)cc2c1=O)ccc3', 
#     MMFF_optimize=True  # 启用MMFF力场优化
# )

rdkit_mol = embed_conformer_from_smiles(
    'CC', 
    MMFF_optimize=True  # 启用MMFF力场优化
)

# ===== 提取分子结构信息 =====
# 功能：从RDKit分子对象中提取原子类型和三维坐标信息

# 提取原子序数数组
# 算法：遍历分子中所有原子，获取其原子序数（元素类型标识）
# 输出：numpy数组，包含每个原子的原子序数
atoms = np.array([a.GetAtomicNum() for a in rdkit_mol.GetAtoms()])

# 提取原子三维坐标
# 算法：从分子构象对象中获取所有原子的笛卡尔坐标
# 输出：numpy数组，形状为(n_atoms, 3)，包含每个原子的x,y,z坐标
positions = rdkit_mol.GetConformer().GetPositions()

# ==================== 创建构象评估对象 ====================
# ConfEval: 单分子构象评估的核心类，负责计算分子的各种化学和物理性质
# 
# 关键参数说明：
# atoms: numpy数组，包含分子中每个原子的原子序数（如C=6, N=7, O=8等）
# positions: numpy数组，形状为(n_atoms, 3)，包含每个原子的3D坐标(x,y,z)
# solvent: 字符串，指定溶剂环境
#   - 'water': 水溶液环境，考虑溶剂化效应
#   - None: 气相环境，不考虑溶剂影响
#   - 其他溶剂名称: 对应的溶剂环境
# 
# 该对象将用于：
# 1. 分子有效性验证（化学结构合理性）
# 2. 分子性质计算（SA分数、QED、logP、fsp3等）
# 3. 构象优化和应变能计算
# 4. 分子指纹生成和相似性比较
conf_eval = ConfEval(atoms, positions, solvent='water')

# 将评估对象的属性转换为pandas Series格式显示
# 这里显示了ConfEval对象的所有可用属性，包括：
# - xyz_block: XYZ格式的分子坐标
# - mol: RDKit分子对象
# - is_valid: 分子结构是否有效
# - SA_score: 合成可达性评分
# - QED: 类药性评分
# - logP: 脂水分配系数
# - fsp3: sp3碳原子比例
# - strain_energy: 应变能
# - rmsd: 均方根偏差
conf_eval.to_pandas()

xyz_block                   8\n\nC -0.75 -0.071 -0.066\nC 0.75 0.071 0.066...
mol                          <rdkit.Chem.rdchem.Mol object at 0x7f81c44d8d60>
smiles                                                                   None
molblock                                                                 None
energy                                                                   None
partial_charges                                                          None
solvent                                                                 water
charge                                                                      0
xyz_block_post_opt                                                       None
mol_post_opt                                                             None
smiles_post_opt                                                          None
molblock_post_opt                                                        None
energy_post_opt                                                 

### 评估模型的整体生成质量

In [ ]:
# ==================== 测试数据准备 ====================
# 定义测试用的简单烷烃分子SMILES列表
# smiles_ls: 包含不同链长烷烃的SMILES字符串列表
#   - 'CC': 乙烷（2个碳原子）
#   - 'CCC': 丙烷（3个碳原子）
#   - 'CCCC': 丁烷（4个碳原子）
smiles_ls = ['CC', 'CCC', 'CCCC']

# 为每个SMILES字符串生成3D构象
# embed_conformer_from_smiles: 核心构象生成函数
#   - 输入: SMILES字符串
#   - MMFF_optimize=False: 不使用MMFF94力场进行后优化，保持ETKDG生成的原始构象
#   - 输出: RDKit分子对象，包含3D坐标信息
# test_mols: RDKit分子对象列表，每个对象包含原子信息和3D构象
test_mols = [embed_conformer_from_smiles(smi, MMFF_optimize=False) for smi in smiles_ls]

# ==================== 数据格式转换 ====================
# 将RDKit分子对象转换为评估管道所需的标准格式
# generated_mols: 评估管道的标准输入格式
#   - 数据结构: List[Tuple[np.ndarray, np.ndarray]]
#   - 每个元组包含: (原子序数数组, 原子坐标矩阵)

generated_mols = []
for m in test_mols:
    # atoms_array: 一维numpy数组，包含分子中每个原子的原子序数
    #   - 例如: [6, 6, 1, 1, 1, 1, 1, 1] 表示乙烷的原子序数
    atoms_array = np.array([a.GetAtomicNum() for a in m.GetAtoms()])
    
    # positions_matrix: 二维numpy数组，形状为(n_atoms, 3)
    #   - 包含每个原子的3D坐标(x, y, z)，单位为埃(Å)
    positions_matrix = m.GetConformer().GetPositions()
    
    # 将原子信息和坐标信息打包为元组，添加到生成分子列表
    generated_mols.append((atoms_array, positions_matrix))

In [29]:
# ==================== 无条件评估管道核心算法 ====================
# UnconditionalEvalPipeline: 无条件分子评估的核心管道类
# 
# 算法实现原理：
# 1. 批量处理架构：采用迭代器模式，逐个处理生成的分子
# 2. 多层次评估体系：
#    - 第一层：分子有效性验证（化学结构合理性检查）
#    - 第二层：分子性质计算（SA分数、QED、logP、fsp3等药物相关性质）
#    - 第三层：构象优化和应变能分析（xTB量子化学计算）
#    - 第四层：分子指纹生成和多样性分析
# 3. 错误处理机制：对无效分子进行标记，不中断整体评估流程
# 4. 结果聚合：统计全局指标（有效率、多样性、平均性质等）
# 
# 关键参数说明：
# generated_mols: List[Tuple[np.ndarray, np.ndarray]] - 待评估的分子列表
# solvent: str - 溶剂环境，影响分子性质计算和构象优化
uncond_pipe = UnconditionalEvalPipeline(generated_mols=generated_mols, solvent='water')

# ==================== 执行批量评估算法 ====================
# evaluate方法的核心工作流程：
# 1. 初始化：创建ConfEval对象池，准备并行计算环境
# 2. 迭代处理：
#    for each molecule in generated_mols:
#        a. 创建ConfEval实例
#        b. 执行分子验证（RDKit分子对象构建）
#        c. 计算分子性质（SA、QED、logP、fsp3）
#        d. 执行构象优化（xTB计算）
#        e. 计算应变能和RMSD
#        f. 生成分子指纹
# 3. 结果汇总：收集所有评估结果，计算统计指标
# 4. 质量控制：验证结果完整性，处理异常情况
# 
# verbose=True: 启用详细输出模式，显示评估进度和中间结果
uncond_pipe.evaluate(verbose=True)

无条件分子评估: 100%|██████████| 3/3 [00:00<00:00,  6.03it/s]


In [30]:
# 将评估管道的结果转换为pandas格式进行分析
# 返回两个对象：
# - properties_df: 包含每个分子详细属性的DataFrame
# - global_attr: 包含全局统计信息的Series
# 
# 注意：此处出现"All arrays must be of the same length"错误
# 这通常是由于评估过程中某些分子的属性数组长度不一致导致的
# 可能的原因包括：分子验证失败、属性计算异常等
properties_df, global_attr = uncond_pipe.to_pandas()

======== 调试信息：检查各属性数组长度 ========
属性 'generated_mols' 的长度: 3
属性 'molblocks' 的长度: 0
属性 'molblocks_post_opt' 的长度: 0
属性 'strain_energies' 的长度: 3
属性 'rmsds' 的长度: 3
属性 'SA_scores' 的长度: 3
属性 'logPs' 的长度: 3
属性 'QEDs' 的长度: 3
属性 'fsp3s' 的长度: 3
属性 'strain_energies_post_opt' 的长度: 3
属性 'rmsds_post_opt' 的长度: 3
属性 'SA_scores_post_opt' 的长度: 3
属性 'logPs_post_opt' 的长度: 3
属性 'QEDs_post_opt' 的长度: 3
属性 'fsp3s_post_opt' 的长度: 3


ValueError: All arrays must be of the same length